# Dimension Probe: does the U-Net memorize at lower intrinsic dimension?

**Motivation.** `edm_unet_memorization_transition.ipynb` found *no* memorization anywhere in a
5x8 grid (n_train ∈ {2..32} x steps ∈ {250..30000}): pixel collapse fraction 0.00 in all 40
cells, while the GMM run through the identical sampler/metric/latents collapses completely
(fraction 1.00, rel NN dist 0.002). So the null is controlled, not a pipeline failure.

**Hypothesis.** Baptista et al.'s U-Net memorization demo uses N=2 *binary* 64x64 squares —
4096 dims, two discrete images. This project's data is 128x128 *continuous* multiscale Gaussian
fields with ~3200 active Fourier modes. The controlling variable is likely the **intrinsic
dimension of the data manifold** (number of active modes), not capacity or training time.

**Design.** Hold the grid at 128x128 — so the U-Net, sampler, and metric are byte-for-byte the
ones already validated — and vary only the spectral support of the data. The config *names*
are historical shorthand; the active-mode counts below are the ones the code actually computes
and stores (`active_modes` in the saved `.pt`):

| config | bands | active modes |
|---|---|---|
| `d12`  | coarse (0.5, 2) | **8** |
| `d50`  | coarse (0.5, 4) | **44** |
| `d314` | coarse (0.5,4) + mid1 (4,10) | **304** |
| `full` | all four bands, k < 32 | **3204** (already run — the null) |

*Why not simply shrink the grid?* Proportionally scaled bands push the coarse band below the
grid's fundamental mode (the smallest nonzero radial wavenumber on any grid is 1), so at 32x32
the coarse band contains **zero** modes: the component generates as all-zeros and the band score
becomes an empty mean — a silent NaN rather than an error (`RingMetricContext` does not guard
this; the assert in the config cell below is what catches it here). At 64x64 it retains only 8
modes. Varying spectral support at fixed grid avoids this entirely and is the cleaner control.

**Primary metrics** are the band-independent ones — pixel-space collapse fraction and median
relative NN distance — since they are directly comparable across configs with different bands.
Each config gets its own GMM reference (the memorization ceiling for that data) **and** its own
fresh-sample baseline (what perfect generalization scores, which on the coarse band is ~0.89–0.94,
not 1.0).

n_train = 2 throughout: the most memorization-prone setting, and the one Baptista uses.
Runtime ~1.3 h.

In [ ]:
import sys, os, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask, make_radial_k_grid
from memorization_metrics import RingMetricContext
from edm import EDMPrecond, EDMScoreWrapper, train_edm
from unet import SmallUNet
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

N = 128
N_TRAIN = 2
TOTAL_STEPS = 30000
CHECKPOINT_AT = [250, 1000, 4000, 8000, 16000, 30000]
BATCH_SIZE = 8

SMOKE = False
if SMOKE:
    TOTAL_STEPS = 200; CHECKPOINT_AT = [100, 200]

results_dir = os.path.join(repo_root, 'results', 'data')
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(results_dir, exist_ok=True); os.makedirs(fig_dir, exist_ok=True)
ckpt_path = os.path.join(results_dir, 'edm_unet_dimension_probe.pt')

In [ ]:
# -- Data configs: same grid, increasing spectral support --
COARSE = {"name": "coarse", "length_scale": 2.0, "s": 2.0, "sigma_sq": 1.0}
MID1   = {"name": "mid1",   "length_scale": 6.0, "s": 2.0, "sigma_sq": 1.0}
MID2   = {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0}
FINE   = {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0}

DATA_CONFIGS = {
    'd12':  dict(components=[{**COARSE, 'band': (0.5, 2.0)}], weights=[1.0]),
    'd50':  dict(components=[{**COARSE, 'band': (0.5, 4.0)}], weights=[1.0]),
    'd314': dict(components=[{**COARSE, 'band': (0.5, 4.0)},
                             {**MID1,   'band': (4.0, 10.0)}], weights=[1.0, 0.8]),
    'full': dict(components=[{**COARSE, 'band': (0.5, 4.0)}, {**MID1, 'band': (4.0, 10.0)},
                             {**MID2, 'band': (10.0, 18.0)}, {**FINE, 'band': (18.0, 32.0)}],
                 weights=[1.0, 0.8, 0.8, 1.2]),
}

kr_ref = make_radial_k_grid(N)
datasets = {}
for name, cfg in DATA_CONFIGS.items():
    res = generate_multiband_dataset_postmask(num_samples=64, grid_size=N,
              components=cfg['components'], weights=cfg['weights'], seed=42, normalize=True)
    bands = res.get('bands', {c['name']: c['band'] for c in cfg['components']})
    kmax = max(b[1] for b in bands.values())
    active = int(((kr_ref < kmax) & (kr_ref > 0)).sum())
    x = res['combined']
    ctx = RingMetricContext(N, bands, device=DEVICE)
    rings = {n: len(ctx.band_rings[n]) for n in bands}
    assert all(r > 0 for r in rings.values()), f'{name}: empty band -> NaN metric: {rings}'
    datasets[name] = dict(x=x, bands=bands, ctx=ctx, active=active)
    print(f'{name:>5}: active modes ~{active:>5}  bands={ {k: tuple(v) for k,v in bands.items()} }  rings={rings}')

In [ ]:
# -- Sampler / evaluation (identical to the transition notebook) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN, N_SDE_STEPS, N_RAND_REF, LATENT_SEED = 16, 1000, 32, 42
if SMOKE:
    N_GEN, N_SDE_STEPS = 4, 50

@torch.no_grad()
def sample_from(score_fn):
    torch.manual_seed(LATENT_SEED)
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    return VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS).reshape(N_GEN, N, N)

@torch.no_grad()
def metrics(x_gen, x_train, ctx):
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    d = torch.cdist(x_gen.flatten(1), x_train.flatten(1))
    nn_rel = (d.min(dim=1).values / x_train.flatten(1).norm(dim=1).mean()).cpu()
    coarse_key = [k for k in ctx.bands if k == 'coarse'][0]
    return {'coarse_score': m[f'{coarse_key}_score'].mean().item(),
            'nn_rel_med': nn_rel.median().item(),
            'collapse_frac': (nn_rel < 0.3).float().mean().item(),
            'mean_ratio': m['mean_ratio'].cpu()}

In [ ]:
# -- Train one UNet per data config (n_train = 2) --
runs = {}
if os.path.exists(ckpt_path):
    runs = torch.load(ckpt_path, map_location='cpu', weights_only=False).get('runs', {})
    print(f'loaded existing: {sorted(runs.keys())}')

for name, d in datasets.items():
    if name in runs:
        print(f'{name}: already trained, skipping'); continue
    print(f'===== training {name} (active modes ~{d["active"]}) =====', flush=True)
    x_train = d['x'][:N_TRAIN].to(DEVICE)
    train_flat = x_train.reshape(N_TRAIN, -1)
    t0 = time.time()
    saved = train_edm(train_flat, grid_size=N, total_steps=TOTAL_STEPS,
                      checkpoint_at=CHECKPOINT_AT, base_channels=16, emb_dim=64,
                      lr=1e-3, batch_size=BATCH_SIZE, seed=0, device=DEVICE,
                      UNetClass=SmallUNet)
    runs[name] = {step: {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
                         'sigma_data': p.sigma_data} for step, p in saved.items()}
    torch.save({'runs': runs}, ckpt_path)
    print(f'  {time.time()-t0:.0f}s', flush=True)

In [ ]:
# -- Evaluate: UNet checkpoints + GMM ceiling, per data config --
evals, gmm_refs = {}, {}
for name, d in datasets.items():
    x_train = d['x'][:N_TRAIN].to(DEVICE)
    train_flat = x_train.reshape(N_TRAIN, -1)
    ctx = d['ctx']
    print(f'===== {name} (active ~{d["active"]}) =====', flush=True)

    gmm = score_models.GMM_score(train_flat, VE_SAMPLE.marginal_prob_mean,
                                 VE_SAMPLE.marginal_prob_std).to(DEVICE)
    gmm_refs[name] = metrics(sample_from(gmm), x_train, ctx)
    g = gmm_refs[name]
    print(f'  GMM ceiling: coarse={g["coarse_score"]:.4f} nn_rel={g["nn_rel_med"]:.4f} '
          f'collapse={g["collapse_frac"]:.2f}')

    evals[name] = {}
    for step, entry in sorted(runs[name].items()):
        unet = SmallUNet(base_channels=16, emb_dim=64).to(DEVICE)
        p = EDMPrecond(unet, sigma_data=entry['sigma_data']).to(DEVICE)
        p.load_state_dict(entry['state_dict']); p.eval()
        w = EDMScoreWrapper(p, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
        r = metrics(sample_from(w), x_train, ctx)
        evals[name][step] = r
        print(f'  step {step:>6}: coarse={r["coarse_score"]:.4f} '
              f'nn_rel={r["nn_rel_med"]:.3f} collapse={r["collapse_frac"]:.2f}', flush=True)

torch.save({'runs': runs, 'evals': evals, 'gmm_refs': gmm_refs,
            'active_modes': {k: v['active'] for k, v in datasets.items()},
            'n_train': N_TRAIN, 'checkpoint_at': CHECKPOINT_AT,
            'note': ('Dimension probe at fixed 128x128: intrinsic dimension varied via spectral '
                     'support (~12/50/314/3205 active modes), n_train=2, matched sampler. Tests '
                     'whether the UNet memorization null is controlled by data dimension.')},
           ckpt_path)
print('saved ->', ckpt_path)

In [ ]:
# -- Plot: memorization vs intrinsic dimension --
# The coarse-band panel needs its own per-config generalization baseline: the NN is the
# minimum over n_train coarse-band distances, so held-out REAL fields do not score 1.0.
# (d50/d314/full share a baseline because their coarse bands are identical by construction —
# the extra components are band-disjoint, and the global normalization is a scale factor
# that cancels in the ratio. That agreement is itself a check on the metric.)
names = list(datasets.keys())
baseline = {n: datasets[n]['ctx'].fresh_baseline(
                  datasets[n]['x'][8:].to(DEVICE), datasets[n]['x'][:N_TRAIN].to(DEVICE),
                  n_rand_ref=N_RAND_REF, exclude_nn=True, n_blocks=3)
            for n in names}
print('generalization baseline (held-out real data), coarse band:')
for n in names:
    print(f'  {n:>5} (~{datasets[n]["active"]} modes): '
          f'{baseline[n]["coarse"][0]:.4f} ± {baseline[n]["coarse"][1]:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
cmap = plt.cm.viridis
colors = {n: cmap(i / max(len(names)-1, 1)) for i, n in enumerate(names)}

for ax, key, title, logy in [
        (axes[0], 'collapse_frac', 'pixel collapse fraction (rel NN dist < 0.3)', False),
        (axes[1], 'nn_rel_med', 'median rel NN distance', True),
        (axes[2], 'coarse_score', 'coarse band score', False)]:
    for n in names:
        steps = sorted(evals[n].keys())
        ax.plot(steps, [evals[n][s][key] for s in steps], marker='o', ms=4,
                color=colors[n], label=f'{n} (~{datasets[n]["active"]} modes)')
        ax.axhline(gmm_refs[n][key], color=colors[n], lw=0.8, ls=':')
        if key == 'coarse_score':
            ax.plot([steps[0], steps[-1]], [baseline[n]['coarse'][0]] * 2,
                    color=colors[n], lw=1.3, ls='--', alpha=0.8)
    ax.set_xscale('log')
    if logy: ax.set_yscale('log')
    ax.set_xlabel('training step'); ax.set_title(title)
axes[2].set_title('coarse band score\ndashed = held-out real data, dotted = GMM', fontsize=9.5)
axes[2].set_ylim(-0.03, 1.05)
axes[0].legend(fontsize=8)
fig.suptitle('Does lowering intrinsic dimension produce U-Net memorization? '
             '(n_train=2; dotted = that config\'s GMM ceiling)', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'unet_dimension_probe.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"config":>6} {"modes":>7} | {"UNet collapse":>13} {"UNet nn_rel":>11} | '
      f'{"GMM collapse":>12} {"GMM nn_rel":>10} | {"coarse vs base":>14}')
print('-' * 88)
for n in names:
    last = evals[n][max(evals[n].keys())]
    print(f'{n:>6} {datasets[n]["active"]:>7} | {last["collapse_frac"]:>13.2f} '
          f'{last["nn_rel_med"]:>11.3f} | {gmm_refs[n]["collapse_frac"]:>12.2f} '
          f'{gmm_refs[n]["nn_rel_med"]:>10.4f} | '
          f'{last["coarse_score"] - baseline[n]["coarse"][0]:>+14.4f}')

## Reading this

- **If collapse fraction rises as active modes fall**, dimension is the controlling variable:
  we have located the boundary of the memorization regime, and both the transition study and the
  covariance-Tikhonov experiment should be re-run at the low-dimension setting where memorization
  actually occurs.
- **If it stays 0.00 even at ~12 active modes** — where the data manifold is essentially a
  12-dimensional Gaussian and the GMM memorizes trivially — then dimension is *not* the
  explanation, and the null is a property of the EDM U-Net + continuous Gaussian data rather
  than of problem size. That is a stronger and more interesting claim, and it points the next
  investigation at the architecture (the amplifier-vs-classifier mechanism) rather than at scale.
- Either way the GMM ceiling per config is the control: it must show collapse ~1.00 for the
  comparison to mean anything.